# 02c — The Manifold Hypothesis

## The One-Sentence Version
Real-world high-dimensional data lies on or near low-dimensional manifolds — and that's the only reason any of this works.

## What You'll Build Intuition For
- What a manifold is (intuitively)
- Why the manifold hypothesis makes the curse survivable
- Visual evidence from real and synthetic data

## Prerequisites
- [02a — Correlation and Redundancy](02a_correlation_and_redundancy.ipynb)
- [02b — Intrinsic vs Ambient Dimension](02b_intrinsic_vs_ambient.ipynb)

## The Story

In 01b we met the curse of dimensionality — high-dimensional space is vast, empty, and hostile. In 02b we saw that real data usually has much lower intrinsic dimension than ambient dimension. Now we connect these two ideas.

The **manifold hypothesis** says: real-world high-dimensional data doesn't fill the entire space. It concentrates on or near a low-dimensional surface — a *manifold* — embedded in that space.

Think of it this way: the curse says "you can never fill a 10,000-dimensional room with data." The manifold hypothesis says "you don't need to — your data isn't in the room. It's on a thin surface inside the room."

If the hypothesis is true (and it is, for virtually all real data), then the curse applies to the *manifold's* dimension, not the ambient dimension. And that changes everything.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_swiss_roll_data, make_digit_data

apply_style()
rng = np.random.default_rng(42)

## What's a Manifold?

A manifold is a smooth surface that might be curved, twisted, or folded — but is **locally flat**. If you zoom in close enough, it looks like a flat plane.

Examples you already know:
- The **surface of the Earth** is a 2D manifold in 3D space. Locally, it looks flat (your kitchen floor). Globally, it's curved (a sphere).
- A **garden hose** is a 1D manifold in 3D space. Water flows along one direction, but the hose coils through three.
- A **Swiss roll** is a 2D manifold in 3D — a flat surface rolled up like a cake.

The key property: the manifold's intrinsic dimension is lower than the space it lives in.

In [ ]:
# The classic Swiss roll: a 2D manifold embedded in 3D
X_swiss, colour = make_swiss_roll_data(n=1500, noise=0.3, seed=42)

fig_swiss = go.Figure(data=[go.Scatter3d(
    x=X_swiss[:, 0], y=X_swiss[:, 1], z=X_swiss[:, 2],
    mode="markers",
    marker=dict(size=2, color=colour, colorscale="Viridis", opacity=0.7),
)])
fig_swiss.update_layout(
    title="Swiss Roll: a 2D surface rolled up in 3D space",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=700, height=500, margin=dict(l=0, r=0, t=40, b=0),
)
fig_swiss.show()

print(f"Shape: {X_swiss.shape} — 3D ambient space")
print("But the data lives on a 2D surface (the rolled sheet).")
print("\nColour encodes position along the roll — it's the hidden 1D coordinate.")

The colour gradient shows the data's hidden structure. Points nearby in colour are nearby *along the manifold*, even if they look far apart in 3D (because the sheet is rolled up). This is why linear methods like PCA struggle with curved manifolds — they measure straight-line distance, not distance along the surface.

We'll tackle this properly in Module 04 (nonlinear methods).

## The Hypothesis

The **manifold hypothesis** is the claim that real-world high-dimensional data concentrates near a low-dimensional manifold.

This isn't just a convenient assumption — there's strong empirical evidence:
- **PCA works.** If data filled all of high-D space, no small set of components could capture most of the variance.
- **Autoencoders work.** If you can compress 784 pixels to 10 latent dimensions and reconstruct, the data was never really 784D.
- **t-SNE and UMAP reveal structure.** When these tools produce clean clusters in 2D, it means the high-D data had low-D structure to find.

The alternative — that data uniformly fills its ambient space — would make all of these methods fail.

## Why It Matters

The manifold hypothesis resolves the apparent contradiction between the curse and reality:

| | Without Manifold Hypothesis | With Manifold Hypothesis |
|---|---|---|
| The curse says... | High-D space is empty and hostile | Still true for the full space |
| But the data... | Would be uniformly sparse | Is concentrated on a low-D surface |
| So algorithms... | Would fail in high-D | Only need to work on the manifold |
| Effective curse dimension | d_ambient | d_intrinsic << d_ambient |

**The curse is about the container. The manifold hypothesis is about the contents.**

## Seeing It

Let's look at two real examples where the manifold hypothesis is visibly true.

In [ ]:
# Example 1: Swiss roll — PCA vs the true structure
# PCA (linear) can't unroll the manifold

pca_swiss = PCA(n_components=2, random_state=42)
X_swiss_pca = pca_swiss.fit_transform(X_swiss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], s=5, alpha=0.5,
            c=colour, cmap="viridis")
ax1.set_title("PCA projection of Swiss Roll")
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")

# What we WANT: the unrolled version (colour preserves order)
ax2.scatter(colour, X_swiss[:, 1], s=5, alpha=0.5, c=colour, cmap="viridis")
ax2.set_title("Unrolled: position along the manifold")
ax2.set_xlabel("Position along roll (manifold coordinate)")
ax2.set_ylabel("Height")

fig.suptitle("PCA can't unroll a curved manifold — that's what Module 04 is for",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/02c_swiss_roll_pca.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Example 2: Digits — 64D ambient, but clear structure in low-D
data, target, images = make_digit_data()

pca_digits = PCA(random_state=42).fit(data)
cumvar = np.cumsum(pca_digits.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Explained variance curve
ax1.plot(range(1, 65), cumvar, "o-", color=COLOURS["signal"], markersize=3)
ax1.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95%")
n_95 = np.searchsorted(cumvar, 0.95) + 1
ax1.axvline(n_95, color=COLOURS["accent"], linewidth=0.8, alpha=0.4)
ax1.annotate(f"{n_95} components", (n_95, 0.95),
             textcoords="offset points", xytext=(10, -15), fontsize=10)
ax1.set_xlabel("Components")
ax1.set_ylabel("Cumulative Variance")
ax1.set_title("Digits: 64D ambient → ~30 intrinsic")
ax1.legend()
ax1.set_ylim(0, 1.05)

# 2D projection coloured by digit
X_2d = PCA(n_components=2, random_state=42).fit_transform(data)
for digit in range(10):
    mask = target == digit
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], s=8, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title("64D → 2D: digit structure emerges")
ax2.legend(title="Digit", markerscale=3, fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig("visuals/02c_digits_manifold.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"64 pixels, but {n_95} components capture 95% of variance.")
print("The digit manifold is far lower-dimensional than the pixel space.")

The digits dataset is a textbook example of the manifold hypothesis in action. Each digit class forms a cluster (or sub-manifold) in 64D space. Even a crude 2D projection reveals this structure.

Think about *why*: a handwritten "3" can vary in size, slant, thickness, and style — but those are only a handful of independent factors. The other ~50 dimensions in pixel space are constrained by "it has to look like a 3." The data lives on a low-dimensional manifold defined by those few factors of variation.

<details>
<summary><b>The Maths Behind This</b></summary>

Formally, a **manifold** $\mathcal{M}$ of dimension $d$ embedded in $\mathbb{R}^D$ (where $d \ll D$) is a topological space that is locally homeomorphic to $\mathbb{R}^d$. In plain terms: zoom in close enough, and it looks like flat $d$-dimensional space.

The **manifold hypothesis** states that real-world data $\{x_1, \ldots, x_n\} \subset \mathbb{R}^D$ concentrates near such a manifold:

$$x_i \approx f(z_i) + \epsilon_i, \quad z_i \in \mathbb{R}^d, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2 I)$$

where $f: \mathbb{R}^d \to \mathbb{R}^D$ is a smooth embedding function. The goal of dimensionality reduction is to find $f^{-1}$ (or an approximation) — mapping data back from the high-dimensional ambient space to the low-dimensional manifold coordinates.

- **PCA** assumes $f$ is linear: $f(z) = Wz + \mu$
- **t-SNE/UMAP** handle nonlinear $f$ by preserving neighbourhood structure
- **Autoencoders** learn $f$ and $f^{-1}$ jointly as neural networks

</details>

## Where This Matters

**Face recognition:** A 256×256 greyscale face image lives in 65,536-dimensional space. But face images are constrained by anatomy — the manifold of valid faces has perhaps 50-100 dimensions (pose, lighting, expression, identity). That's why face recognition works.

**Drug discovery:** A molecule can be described by thousands of molecular descriptors. But the space of *stable, synthesisable, drug-like molecules* is a tiny manifold within that space. Navigating this manifold is the core of computational drug design.

## Feynman Check

1. **What would it mean if the manifold hypothesis were *wrong*?** (Hint: think about what would happen to PCA, autoencoders, and clustering.)

2. **Why does the manifold hypothesis make the curse of dimensionality less scary?**

3. **Give an intuitive argument for why face images should lie on a low-dimensional manifold.** (Hint: how many independent things can vary in a face?)

---

Module 02 is complete. You now understand *why* dimensionality reduction is possible: data has redundancy (02a), intrinsic dimension is lower than ambient dimension (02b), and real data concentrates on low-dimensional manifolds (02c).

In **Module 03: Linear Methods**, we start learning *how* to actually do it — beginning with PCA.